# House Prices Dataset
[kaggle](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)

In [4]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import interact
%matplotlib inline

In [16]:
import torch
from torch import nn

In [5]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

In [8]:
df = pd.read_csv(path + '/train.csv')
df_test = pd.read_csv(path + '/test.csv')
df

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [17]:
X = df[[
    'LotArea', 'YearBuilt', 'GrLivArea', 'FullBath', 'BedroomAbvGr',
    'KitchenAbvGr', 'GarageArea', 'OpenPorchSF', 'PoolArea', 'YrSold'
]]
y = df[['SalePrice']]

print(y)
X

      SalePrice
0        208500
1        181500
2        223500
3        140000
4        250000
...         ...
1455     175000
1456     210000
1457     266500
1458     142125
1459     147500

[1460 rows x 1 columns]


,LotArea,YearBuilt,GrLivArea,FullBath,BedroomAbvGr,KitchenAbvGr,GarageArea,OpenPorchSF,PoolArea,YrSold
0,8450,2003,1710,2,3,1,548,61,0,2008
1,9600,1976,1262,2,3,1,460,0,0,2007
2,11250,2001,1786,2,3,1,608,42,0,2008
3,9550,1915,1717,1,3,1,642,35,0,2006
4,14260,2000,2198,2,4,1,836,84,0,2008
...,...,...,...,...,...,...,...,...,...,...
1455,7917,1999,1647,2,3,1,460,40,0,2007
1456,13175,1978,2073,2,3,1,500,0,0,2010
1457,9042,1941,2340,2,4,1,252,60,0,2010
1458,9717,1950,1078,1,2,1,240,0,0,2010


In [53]:
A, B = X.mean(), X.std()
C, D = y.mean(), y.std()

X_normalized = (X - A) / B

X_normalized

,LotArea,YearBuilt,GrLivArea,FullBath,BedroomAbvGr,KitchenAbvGr,GarageArea,OpenPorchSF,PoolArea,YrSold
0,-0.207071,1.050634,0.370207,0.789470,0.163723,-0.211381,0.350880,0.216429,-0.068668,0.138730
1,-0.091855,0.156680,-0.482347,0.789470,0.163723,-0.211381,-0.060710,-0.704242,-0.068668,-0.614228
2,0.073455,0.984415,0.514836,0.789470,0.163723,-0.211381,0.631510,-0.070337,-0.068668,0.138730
3,-0.096864,-1.862993,0.383528,-1.025689,0.163723,-0.211381,0.790533,-0.175988,-0.068668,-1.367186
4,0.375020,0.951306,1.298881,0.789470,1.389547,-0.211381,1.697903,0.563567,-0.068668,0.138730
...,...,...,...,...,...,...,...,...,...,...
1455,-0.260471,0.918196,0.250316,0.789470,0.163723,-0.211381,-0.060710,-0.100523,-0.068668,-0.614228
1456,0.266316,0.222899,1.061003,0.789470,0.163723,-0.211381,0.126376,-0.704242,-0.068668,1.644646
1457,-0.147760,-1.002149,1.569110,0.789470,1.389547,-0.211381,-1.033560,0.201336,-0.068668,1.644646
1458,-0.080133,-0.704164,-0.832502,-1.025689,-1.062101,-0.211381,-1.089686,-0.704242,-0.068668,1.644646


In [130]:
X_tensor = torch.tensor(X_normalized.values, dtype=torch.float32)
y_tensor = torch.tensor(y.values, dtype=torch.float32)

model = nn.Linear(10, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.01)

for _ in range(1_000):
    loss = nn.functional.mse_loss(model(X_tensor), y_tensor)
    opt.zero_grad()
    loss.backward()
    opt.step()

    with torch.no_grad():
        model.weight.clamp_(min=0)

In [131]:
model.weight, model.bias

(Parameter containing:
 tensor([[ 6614.7031, 24886.6758, 41061.7891,     0.0000,     0.0000,     0.0000,
          16616.2520,  2278.1162,     0.0000,   221.5791]], requires_grad=True),
 Parameter containing:
 tensor([180920.8125], requires_grad=True))

In [132]:
y_pred = model(X_tensor)

mae = (y_tensor - y_pred).abs().mean()
print(f'Mean Absolute Error ($): {mae:.2f}')

mse = ((y_tensor - y_pred) ** 2).mean()
print(f'Mean Squared Error ($^2): {mse:.2f}')

Mean Absolute Error ($): 28593.16
Mean Squared Error ($^2): 1946045056.00


In [133]:
weights_df = pd.DataFrame([
    {'Column': col, 'Weight': round(float(weight))}
    for col, weight in zip(X.columns, model.weight[0].detach())
])
weights_df

,Column,Weight
0,LotArea,6615
1,YearBuilt,24887
2,GrLivArea,41062
3,FullBath,0
4,BedroomAbvGr,0
5,KitchenAbvGr,0
6,GarageArea,16616
7,OpenPorchSF,2278
8,PoolArea,0
9,YrSold,222


`Ground Living Area` i `Year Built` są najbardziej znaczącymi kolumnami